# PROG8245: Inverted Matrix & Information Retrieval Workshop

**Class:** PROG8245 - Applied AI & Machine Learning  
**Assignment:** Inverted Index, Positional Indexes, and TF-IDF Implementation  
**Date:** November 24, 2026  

---

### 👥 Team 3

| Name | Student ID |
| :--- | :--- |
| Ricardo Mohammed | 7500382 |
| Juan Camilo Chirivi | 9115141 |
| Zeynep Ozdemir | 9045142 |
| Senay Teweldebrhan | 9120588 |

---

### 📝 Project Overview
This notebook implements a complete Information Retrieval pipeline from scratch, including document collection, linguistic preprocessing, inverted and positional indexing, Boolean/phrase queries, and TF-IDF document ranking. Finally, we explore

In [13]:
import nltk
import re
import math
from collections import defaultdict
import pandas as pd

# Download necessary NLTK data packages
nltk.download('movie_reviews', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True) # Required for some NLTK tokenizers

from nltk.corpus import movie_reviews, stopwords
from nltk.stem import PorterStemmer

# Initialize our tools
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

## Step 1: Document Collection and Tokenization

### 📚 Theory: The Corpus and Vocabulary
In Information Retrieval (IR), the first step is gathering a **corpus**—a large, structured set of texts. To build a robust Inverted Index and calculate meaningful TF-IDF scores, our corpus must be large enough to capture a diverse vocabulary. A vocabulary of over 2,000 unique words ensures that our Inverse Document Frequency (IDF) calculations have enough statistical variance to properly distinguish between common and rare terms.

### ⚙️ The Collection Process
For this assignment, we are utilizing the **NLTK `movie_reviews` corpus**. 
* **Why this dataset?** It contains thousands of movie reviews categorized by sentiment. It provides a rich, varied vocabulary with natural language syntax, making it ideal for testing text normalization and indexing.
* **Implementation:** Instead of loading the entire 2,000-document corpus (which can slow down manual indexing), we will load a representative subset of 100 documents. This easily exceeds the 2,000 unique word threshold while keeping execution times fast for iterative testing.

> 💡 **Original Talking Point (Data Collection):** 
> *While a larger corpus improves the statistical reliability of IDF calculations, it introduces a computational bottleneck during the indexing phase. Furthermore, using a general-purpose stop-word list on a specialized corpus (like medical or legal texts) might accidentally strip out domain-critical terms that act as 'stop words' in everyday English but are highly informative in context. For a production AI Agent, we would need to generate custom, domain-specific stop-word lists.*

In [14]:
# Load 100 movie reviews to ensure we easily exceed 2,000 unique words
# (The full corpus has 2,000 reviews, but 100 is plenty for this vocab size)
documents = [movie_reviews.raw(fileid) for fileid in movie_reviews.fileids()[:100]]

print(f"Successfully loaded {len(documents)} documents.")
print(f"Preview of Document 0 (first 150 chars):\n{documents[0][:150]}...")

Successfully loaded 100 documents.
Preview of Document 0 (first 150 chars):
plot : two teen couples go to a church party , drink and then drive . 
they get into an accident . 
one of the guys dies , but his girlfriend continue...


## Step 2: Linguistic Processing Pipeline (Normalization & Stemming)

### 📚 Theory: Why Preprocess Text?
Raw text is highly unstructured and "noisy." If we feed raw text directly into an Inverted Index, the system will treat "Run", "run", "running", and "ran" as four completely different terms. This creates a **sparse matrix**, dilutes term frequencies, and severely degrades search recall. 

To solve this, we implement a **Linguistic Processing Pipeline** to reduce dimensionality and standardize the text.

### ⚙️ The Algorithms
Our pipeline applies four sequential transformations:
1. **Normalization (Lowercasing):** Converts all characters to lowercase so that "The" and "the" map to the same token.
2. **Tokenization (Regex Extraction):** We use the regular expression `\b\w+\b` to extract only alphanumeric words. This effectively strips away punctuation, special characters, and formatting noise.
3. **Stop-Word Removal:** Filters out high-frequency, low-information words (e.g., "the", "is", "and") using the NLTK English stop-word list. This drastically reduces the size of our inverted index.
4. **Affix Stripping (Stemming):** We apply the **Porter Stemmer** algorithm. Stemming uses heuristic rules to chop off common suffixes (like "-ing", "-es", "-tion") to reduce words to their morphological root form.

> 💡 **Original Talking Point (Stemming vs. Lemmatization in AI):** 
> *While stemming (like the Porter algorithm) is computationally cheap and highly effective for traditional Boolean and TF-IDF retrieval, it often produces non-linguistic root words (e.g., "university" becomes "univers"). In modern AI Agents and RAG (Retrieval-Augmented Generation) pipelines, we often prefer **lemmatization**—which uses vocabulary and morphological analysis to return actual dictionary roots (e.g., "better" becomes "good"). Lemmatization preserves semantic meaning for Large Language Models (LLMs), even though stemming remains the standard for fast, keyword-based search engines.*

In [15]:
def preprocess(text):
    """
    Tokenizes and normalizes text by:
    1. Lowercasing
    2. Extracting only alphanumeric words (removes punctuation)
    3. Removing English stop words
    4. Applying Porter Stemming (affix stripping)
    """
    # 1. Lowercase
    text = text.lower()
    
    # 2. Tokenize & remove punctuation (keep only words)
    words = re.findall(r'\b\w+\b', text)
    
    # 3. Remove stop words
    words = [w for w in words if w not in stop_words]
    
    # 4. Stemming (Affix stripping)
    words = [stemmer.stem(w) for w in words]
    
    return words

# Test the pipeline on the first document
print("Processed tokens from Doc 0 (first 20):")
print(preprocess(documents[0])[:20])

Processed tokens from Doc 0 (first 20):
['plot', 'two', 'teen', 'coupl', 'go', 'church', 'parti', 'drink', 'drive', 'get', 'accid', 'one', 'guy', 'die', 'girlfriend', 'continu', 'see', 'life', 'nightmar', 'deal']


In [16]:
# Collect all unique words across the entire corpus
unique_vocab = set()
for doc in documents:
    unique_vocab.update(preprocess(doc))

print(f"Total unique words in our processed corpus: {len(unique_vocab)}")

if len(unique_vocab) > 2000:
    print("✅ Rubric Requirement Met: Corpus has more than 2,000 unique words.")
else:
    print("⚠️ Warning: Corpus has fewer than 2,000 unique words. Load more documents.")

Total unique words in our processed corpus: 6795
✅ Rubric Requirement Met: Corpus has more than 2,000 unique words.


## Step 3: Inverted Index and Boolean Retrieval

### 📚 Theory: The Inverted Index
An **Inverted Index** is the foundational data structure of modern search engines. Instead of mapping Documents $\rightarrow$ Terms (like a standard dictionary), it flips the relationship to map **Terms $\rightarrow$ Documents**. 

For every unique word in our corpus, we create a **posting list**—a list of Document IDs where that word appears. Because we process our documents sequentially (from ID 0 to 99), our posting lists are naturally sorted in ascending order, which is a critical optimization for fast query execution.

### ⚙️ Boolean Retrieval Logic
Once the index is built, we can execute Boolean queries using basic set theory:
* **AND Query:** Returns documents containing *both* terms. Mathematically, this is the **intersection** ($A \cap B$) of the two posting lists.
* **AND NOT Query:** Returns documents containing the first term but *excluding* the second. Mathematically, this is the **set difference** ($A - B$).

> 💡 **Original Talking Point (Indexing Efficiency):** 
> *While Python's built-in `set` operations make Boolean queries incredibly easy to write, enterprise search engines (like Elasticsearch or Solr) don't just use raw sets. They use compressed data structures like **Roaring Bitmaps** or **Frame of Reference (FOR) encoding** for posting lists. This allows them to perform intersections on millions of documents in milliseconds while keeping the index footprint small enough to fit entirely in RAM, avoiding costly disk I/O bottlenecks.*

In [17]:
def build_inverted_index(docs, preprocess_fn):
    """
    Builds an inverted index mapping stemmed tokens to a list of document IDs.
    """
    index = defaultdict(list)
    
    for doc_id, doc in enumerate(docs):
        # Preprocess the document to get clean, stemmed tokens
        tokens = preprocess_fn(doc)
        
        # Use a set to get unique tokens for this specific document
        # This prevents duplicate doc_ids in the posting list
        unique_tokens = set(tokens)
        
        # Add the doc_id to the posting list for each unique token
        for token in unique_tokens:
            index[token].append(doc_id)
            
    return dict(index)

# Build the index using our 100 movie reviews
print("Building Inverted Index...")
inverted_index = build_inverted_index(documents, preprocess)
print(f"✅ Index built successfully! Total unique terms: {len(inverted_index)}")

Building Inverted Index...
✅ Index built successfully! Total unique terms: 6795


In [18]:
# Helper function to preprocess a raw query term so it matches our stemmed index
def get_posting_list(index, raw_term):
    processed = preprocess(raw_term)
    stemmed_term = processed[0] if processed else raw_term
    return index.get(stemmed_term, [])

# Test 1: Search for a common word
term1 = "movie"
print(f"Documents containing '{term1}' (stemmed to '{preprocess(term1)[0]}'):")
print(f"Count: {len(get_posting_list(inverted_index, term1))} documents")
print(f"Doc IDs: {get_posting_list(inverted_index, term1)[:10]}...\n") # Show first 10

# Test 2: Search for a less common word
term2 = "nasa"
print(f"Documents containing '{term2}':")
print(f"Doc IDs: {get_posting_list(inverted_index, term2)}")

Documents containing 'movie' (stemmed to 'movi'):
Count: 87 documents
Doc IDs: [0, 1, 2, 4, 5, 6, 7, 8, 9, 10]...

Documents containing 'nasa':
Doc IDs: [70, 71]


### 🔍 Implementing Boolean Queries
To execute Boolean logic, we retrieve the posting lists for our query terms and apply set operations. We will implement `boolean_and` (intersection) and `boolean_and_not` (difference).

In [19]:
def boolean_and(index, term1, term2):
    """Returns doc IDs containing BOTH term1 AND term2."""
    list1 = set(get_posting_list(index, term1))
    list2 = set(get_posting_list(index, term2))
    return sorted(list(list1 & list2)) # Intersection

def boolean_and_not(index, term1, term2):
    """Returns doc IDs containing term1 BUT NOT term2."""
    list1 = set(get_posting_list(index, term1))
    list2 = set(get_posting_list(index, term2))
    return sorted(list(list1 - list2)) # Difference

# --- Testing Boolean Queries ---

# Helper to print a snippet of the document so we can verify the results
def print_doc_snippet(doc_id, max_chars=100):
    print(f"  -> Doc {doc_id}: {documents[doc_id][:max_chars].replace(chr(10), ' ')}...")

print("--- TEST 1: AND Query ---")
print("Searching for: 'movi' AND 'great'")
and_results = boolean_and(inverted_index, "movie", "great")
print(f"Found {len(and_results)} documents: {and_results[:5]}")
for doc_id in and_results[:2]:
    print_doc_snippet(doc_id)

print("\n--- TEST 2: AND NOT Query ---")
print("Searching for: 'movi' AND NOT 'bad'")
and_not_results = boolean_and_not(inverted_index, "movie", "bad")
print(f"Found {len(and_not_results)} documents: {and_not_results[:5]}")
for doc_id in and_not_results[:2]:
    print_doc_snippet(doc_id)

--- TEST 1: AND Query ---
Searching for: 'movi' AND 'great'
Found 18 documents: [5, 17, 18, 19, 23]
  -> Doc 5: capsule : in 2176 on the planet mars police taking into custody an accused murderer face the title m...
  -> Doc 17: so what do you get when you mix together plot elements from various successful sci-fi films such as ...

--- TEST 2: AND NOT Query ---
Searching for: 'movi' AND NOT 'bad'
Found 46 documents: [1, 2, 6, 10, 11]
  -> Doc 1: the happy bastard's quick movie review  damn that y2k bug .  it's got a head start in this movie sta...
  -> Doc 2: it is movies like these that make a jaded movie viewer thankful for the invention of the timex indig...


## Step 4: Biword and Positional Indexes

### 📚 Theory: The Problem with Standard Inverted Indexes
A standard inverted index tells us *which* documents contain a term, but it loses the **word order**. If we search for "cat eats dog", a standard index will return documents containing "dog eats cat" because it only checks for the presence of the words, not their sequence. 

To solve this, we use **Positional Indexes** (and their subset, **Biword Indexes**). 
* **Biword Index:** Specifically indexes pairs of consecutive words (bigrams). It's fast for 2-word phrases but the index size explodes if we try to index 3-word or 4-word phrases.
* **Positional Index:** A more flexible approach. It stores the exact numerical position (offset) of every term within a document. 

### ⚙️ Positional Matching Logic
To execute a phrase query (e.g., "bad movie"), we:
1. Retrieve the positional posting lists for "bad" and "movie".
2. Find the **intersection** of documents that contain both words.
3. For each candidate document, check the positions. If "bad" is at position $p$, we check if "movie" exists at position $p + 1$. If the positions are strictly adjacent, we have a phrase match.

> 💡 **Original Talking Point (Positional Indexes in Modern AI):** 
> *While positional indexes are the gold standard for traditional keyword search engines (like Elasticsearch), modern Retrieval-Augmented Generation (RAG) pipelines often rely on Vector Databases for semantic search. However, vector search struggles with exact keyword matching (e.g., searching for a specific product ID or exact legal phrase). Therefore, the best AI agents use a **hybrid approach**: combining dense vector retrieval for semantic meaning with sparse positional indexes for exact phrase matching and keyword filtering.*

In [20]:
def build_positional_index(docs, preprocess_fn):
    """
    Builds a positional index.
    Returns a nested dictionary: {term: {doc_id: [position1, position2, ...]}}
    """
    pos_index = defaultdict(lambda: defaultdict(list))
    
    for doc_id, doc in enumerate(docs):
        tokens = preprocess_fn(doc)
        # Enumerate gives us both the token and its exact position (index) in the document
        for pos, token in enumerate(tokens):
            pos_index[token][doc_id].append(pos)
            
    return pos_index

# Build the positional index for our corpus
print("Building Positional Index...")
positional_index = build_positional_index(documents, preprocess)

# Quick verification: Let's look at the positions for the word "movie" (stemmed to "movi")
print(f"\nPositions for 'movi' in Document 0: {positional_index['movi'].get(0, [])}")

Building Positional Index...

Positions for 'movi' in Document 0: [21, 27, 72, 166, 189, 208, 223]


### 🔍 Implementing Phrase Queries
Now we will write a function that takes a raw text phrase, preprocesses it, and uses the positional index to find documents where the words appear in the exact consecutive order.

In [21]:
def phrase_query(pos_index, raw_phrase, preprocess_fn):
    """
    Executes a phrase query using positional intersection.
    Returns a list of document IDs where the exact phrase appears.
    """
    # 1. Preprocess the query phrase into tokens
    query_tokens = preprocess_fn(raw_phrase)
    
    if len(query_tokens) < 2:
        return "Phrase must be at least 2 words."
    
    # 2. Find candidate documents (Intersection of docs containing ALL terms)
    candidate_docs = set(pos_index[query_tokens[0]].keys())
    for token in query_tokens[1:]:
        if token not in pos_index:
            return [] # If any word is missing from the index, the phrase can't exist
        candidate_docs &= set(pos_index[token].keys())
    
    results = []
    
    # 3. Positional Matching
    for doc_id in candidate_docs:
        # Get all positions of the FIRST word in this document
        positions_first_word = pos_index[query_tokens[0]][doc_id]
        
        for start_pos in positions_first_word:
            match_found = True
            
            # Check if subsequent words appear at consecutive positions (start_pos + 1, start_pos + 2, etc.)
            for i in range(1, len(query_tokens)):
                target_pos = start_pos + i
                
                # Check if target_pos exists in the positions list for the i-th word
                if target_pos not in pos_index[query_tokens[i]][doc_id]:
                    match_found = False
                    break # Stop checking this starting position
            
            if match_found:
                results.append(doc_id)
                break # We found the phrase in this doc, move to the next document
                
    return results

In [22]:
# --- Phrase Query Test 1 ---
# Let's search for a phrase likely to appear in movie reviews: "bad movi" (stemmed from "bad movie")
phrase1 = "bad movie"
results1 = phrase_query(positional_index, phrase1, preprocess)

print(f"--- Phrase Query 1: '{phrase1}' ---")
print(f"Stemmed query tokens: {preprocess(phrase1)}")
print(f"Found in {len(results1)} documents.")
print(f"Document IDs: {results1[:5]}\n")

# --- Phrase Query Test 2 ---
# Let's search for another phrase: "good stori" (stemmed from "good story")
phrase2 = "good story"
results2 = phrase_query(positional_index, phrase2, preprocess)

print(f"--- Phrase Query 2: '{phrase2}' ---")
print(f"Stemmed query tokens: {preprocess(phrase2)}")
print(f"Found in {len(results2)} documents.")
print(f"Document IDs: {results2[:5]}\n")

# --- Verification ---
# Let's print a snippet from the first document of Test 1 to prove the phrase is actually there
if results1:
    doc_id = results1[0]
    print(f"Verification snippet from Doc {doc_id}:")
    print(documents[doc_id][:300].replace('\n', ' '))

--- Phrase Query 1: 'bad movie' ---
Stemmed query tokens: ['bad', 'movi']
Found in 5 documents.
Document IDs: [9, 14, 38, 83, 92]

--- Phrase Query 2: 'good story' ---
Stemmed query tokens: ['good', 'stori']
Found in 2 documents.
Document IDs: [61, 84]

Verification snippet from Doc 9:
plot : a young french boy sees his parents killed before his eyes by tim roth , oops . . . i  mean , an evil man .  he vows revenge on that man and is taught the ways of the musketeer by some old dude who used to be one himself ?  anyway , fourteen years go by and . . . arrgh , well , you know the r


## Step 5: TF-IDF, Document Ranking, and AI Applications

### 📚 Theory: TF-IDF and Document Ranking
While Boolean and Phrase queries tell us *if* a document matches, they don't tell us *how relevant* it is. To rank documents, we use **TF-IDF (Term Frequency-Inverse Document Frequency)**.

* **Term Frequency (TF):** How often a term appears in a specific document. (Higher = more relevant to that doc).
* **Inverse Document Frequency (IDF):** How rare a term is across the entire corpus. (Higher = more informative).
* **TF-IDF Score:** $TF \times IDF$. It rewards terms that appear frequently in a document but rarely across the whole corpus.

**Document Ranking:** When a user submits a query, we calculate the TF-IDF score for every document in the corpus relative to the query terms. We then sort the documents in descending order of their scores. The document with the highest score is ranked #1 because it contains the most "informative" matches for the query.

### ⚙️ Implementation
Instead of relying on a black-box library, we will calculate TF-IDF manually using our existing Positional Index. We can extract the Term Frequency (length of the positions list) and the Document Frequency (number of documents in the index for that term) directly from our data structure.

In [23]:
def rank_documents_by_tfidf(query, pos_index, total_docs, top_n=3):
    """
    Calculates TF-IDF scores for a query and returns the top N ranked documents.
    """
    # 1. Preprocess the query
    query_tokens = preprocess(query)
    
    if not query_tokens:
        return []

    # 2. Calculate IDF for each unique term in the query
    idf_scores = {}
    for token in set(query_tokens):
        df = len(pos_index.get(token, {})) # Number of docs containing the term
        if df > 0:
            # Standard IDF formula: log(N / df)
            idf_scores[token] = math.log(total_docs / df)
        else:
            idf_scores[token] = 0.0
            
    # 3. Calculate TF-IDF score for each document
    doc_scores = defaultdict(float)
    
    for token in query_tokens:
        if token in pos_index:
            for doc_id, positions in pos_index[token].items():
                tf = len(positions) # Term frequency (raw count)
                doc_scores[doc_id] += tf * idf_scores[token]
                
    # 4. Sort documents by score in descending order
    ranked_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
    
    return ranked_docs[:top_n]

# --- Testing the Ranker ---
test_query = "bad movie"
print(f"Ranking documents for query: '{test_query}'\n")

top_results = rank_documents_by_tfidf(test_query, positional_index, len(documents), top_n=3)

for rank, (doc_id, score) in enumerate(top_results, 1):
    print(f"Rank #{rank} | Doc ID: {doc_id} | TF-IDF Score: {score:.4f}")
    print(f"Snippet: {documents[doc_id][:120].replace(chr(10), ' ')}...\n")
    print("-" * 50)

Ranking documents for query: 'bad movie'

Rank #1 | Doc ID: 13 | TF-IDF Score: 5.1868
Snippet: a movie like mortal kombat : annihilation works ( and must be reviewed on ) multiple levels .  first , there's the rampa...

--------------------------------------------------
Rank #2 | Doc ID: 9 | TF-IDF Score: 4.6808
Snippet: plot : a young french boy sees his parents killed before his eyes by tim roth , oops . . . i  mean , an evil man .  he v...

--------------------------------------------------
Rank #3 | Doc ID: 27 | TF-IDF Score: 4.5416
Snippet: reindeer games is easily the worst of the three recent films penned by ehren kruger ( scream 3 and arlington rd are the ...

--------------------------------------------------


### 🤖 Connecting IR to RAG and AI Agents
Traditional Information Retrieval (like the Inverted Index and TF-IDF we just built) is not obsolete; it is a critical component of modern **Retrieval-Augmented Generation (RAG)** and **AI Agents**.

#### 1. The Role of IR in RAG (Hybrid Search)
In a standard RAG pipeline, an LLM generates answers based on retrieved context. While modern RAG uses **Dense Retrieval** (Vector Databases / Embeddings) to find *semantically* similar text, vector search often fails at exact keyword matching (e.g., searching for a specific product ID, a legal clause, or an exact name). 

Therefore, production RAG systems use **Hybrid Search**: combining Dense Retrieval (Vectors) with Sparse Retrieval (TF-IDF / BM25 Inverted Indexes). The Inverted Index ensures exact keyword matches are never missed, while the Vector DB captures conceptual meaning.

#### 2. AI Agents and Tool Calling
AI Agents (like AutoGPT or LangChain agents) operate by reasoning and calling external "tools". When an Agent needs to answer a question about a proprietary company database, it doesn't guess; it uses a **Search Tool**. Under the hood, that Search Tool is often an optimized Inverted Index executing a Boolean or TF-IDF query to fetch the precise context the Agent needs to formulate its final response.

> 💡 **Original Talking Point (The Future of IR in AI):** 
> *As AI Agents become more autonomous, the "retrieval" step is shifting from a simple search to an agentic workflow. Instead of just returning a list of documents, future IR systems will act as "Research Agents" that iteratively refine their own Boolean and TF-IDF queries based on the LLM's intermediate reasoning steps. The Inverted Index remains the fast, deterministic backbone that prevents the LLM from hallucinating by grounding it in exact, verifiable data.*

## Summary and Final Reflection

**Term Frequency (TF) and Inverse Document Frequency (IDF)**

| Term | What is it? | How is it used? | Pros | Cons |
| --- | --- | --- | --- | --- |
| **Term Frequency (TF)** | The number of times a term appears in one document. Our code uses the raw count. | Gives weight to repetition. Our code obtains TF from the length of the term's position list in a document. | Simple to calculate; reuses the positional index; captures repetition within a document. | Raw counts can favor longer documents; repetition does not always mean relevance; ignores how common the term is across documents. |
| **Inverse Document Frequency (IDF weight)** | A weight calculated as `ln(N / df)`, where `N` is the total number of documents and `df` is the number containing the term. | Gives more weight to query terms found in fewer documents. It is multiplied by TF when calculating relevance scores. | Reduces the influence of widespread terms; helps distinguish documents using less common query terms. | Rare terms may be typos or irrelevant; depends on the selected collection; does not capture meaning or word order. |


**Reflection**

This workshop helped us understand how a search engine organizes text before answering a query. The second request used a total of 100 movie reviews and 6,795 unique terms were obtained after processing. Size reduction, tokenization, deletion of keywords and stemming helped us to process some variations of words consistently. 

The inverted index helped us find documents that contained specific words, and boolean operations allowed us to combine or exclude terms. The positional index added information about word order, allowing us to search for sequences like "bad movie" and "good story." One limitation is that our positions belong to the processed text. TF-IDF helped us move from finding matching documents to assigning keyword relevance scores. TF gives weight to repeated terms, while IDF reduces the weight of terms shared by many documents. In general, the workshop connected text preparation, indexing and classification.



